# NumPy: Cómputo vectorizado, broadcasting y por qué los loops son lentos

**Ciencia de Datos, Sección A** · Sesión 2 · 23 de julio de 2026

Notebook companion de la presentación. Corran cada celda con `Shift+Enter`.

## 1. ¿Por qué NumPy?

Una lista de Python guarda **punteros a objetos**; un `ndarray` es un **bloque contiguo de memoria** con un solo tipo. Midamos la diferencia:

In [1]:
import numpy as np

datos_lista = list(range(10_000_000))
datos_array = np.arange(10_000_000)

In [2]:
%timeit sum(datos_lista)

40.6 ms ± 236 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [3]:
%timeit datos_array.sum()

1.03 ms ± 14.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 2. El ndarray: creación, dtype y shape

In [4]:
a = np.array([1, 2, 3, 4])
print(a.dtype, a.shape)

m = np.array([[1, 2, 3],
              [4, 5, 6]])
print(m.shape, m.ndim)

x = np.array([1.0, 2, 3])
print(x.dtype)  # promoción automática a float64

int64 (4,)
(2, 3) 2
float64


In [5]:
print(np.zeros((3, 4)))
print(np.ones(5))
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[1. 1. 1. 1. 1.]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


In [6]:
# Reproducibilidad explícita: default_rng(seed), no np.random.seed()
rng = np.random.default_rng(seed=42)
print(rng.normal(size=(2, 3)))
print(rng.integers(1, 7, size=10))  # 10 tiros de dado

[[ 0.30471708 -1.03998411  0.7504512 ]
 [ 0.94056472 -1.95103519 -1.30217951]]
[5 5 5 5 4 1 6 3 4 3]


## 3. Vectorización: operar sobre el array completo

Las **ufuncs** reemplazan al ciclo `for`: el ciclo corre en C, no en Python.

In [7]:
precios = np.array([100.0, 250.0, 80.0, 120.0])

print(precios * 1.12)             # IVA
print(precios - precios.mean())   # centrar en la media
print(np.log(precios))            # ufunc elemento a elemento
print(precios > 100)              # comparación vectorizada

[112.  280.   89.6 134.4]
[-37.5 112.5 -57.5 -17.5]
[4.60517019 5.52146092 4.38202663 4.78749174]
[False  True False  True]


## 4. Indexing y slicing

In [8]:
a = np.arange(10)
print(a[2:7])    # [2 3 4 5 6]
print(a[::2])    # pares de índice
print(a[::-1])   # invertido

m = np.arange(12).reshape(3, 4)
print(m)
print(m[0, :])    # primera fila
print(m[:, 1])    # segunda columna
print(m[1:, :2])  # submatriz

[2 3 4 5 6]
[0 2 4 6 8]
[9 8 7 6 5 4 3 2 1 0]
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[0 1 2 3]
[1 5 9]
[[4 5]
 [8 9]]


In [9]:
notas = np.array([45, 78, 92, 60, 55, 88])

mascara = notas >= 61
print(mascara)
print(notas[mascara])
print(notas[notas >= 61].mean())  # promedio de aprobados

# Combinar condiciones: & y |, siempre con paréntesis
print(notas[(notas >= 61) & (notas < 90)])

[False  True  True False False  True]
[78 92 88]
86.0
[78 88]


In [10]:
# ¡Cuidado! El slicing devuelve una VISTA, no una copia
a = np.arange(5)
b = a[1:4]
b[0] = 99
print(a)  # ¡a cambió!

c = a[1:4].copy()  # copia independiente
c[0] = -1
print(a)  # a no cambia

[ 0 99  2  3  4]
[ 0 99  2  3  4]


## 5. Broadcasting

Reglas: alinear shapes **desde la derecha**; dimensiones iguales son compatibles; una dimensión de 1 **se estira** (sin copiar memoria); si no, error.

| A | B | Resultado |
|---|---|---|
| `(3, 4)` | `(4,)` | `(3, 4)` |
| `(3, 1)` | `(1, 4)` | `(3, 4)` |
| `(3, 4)` | `(3,)` | error |

In [18]:
ventas = np.array([[10, 20, 30],     # tienda A
                   [40, 50, 60]])    # tienda B

print(ventas * 2)                    # escalar

            # (2,3) - (2,1) -> por fila

[[ 20  40  60]
 [ 80 100 120]]


In [19]:
precio = np.array([1.5, 2.0, 0.5])   # por producto
print(ventas * precio)               # (2,3) * (3,) -> por columna

[[ 15.  40.  15.]
 [ 60. 100.  30.]]


In [20]:
meta = np.array([[25], [45]])        # por tienda
print(ventas - meta)   

[[-15  -5   5]
 [ -5   5  15]]


In [12]:
ventas = np.array([[10, 20, 30],     # tienda A
                   [40, 50, 60]])    # tienda B

print(ventas * 2)                    # escalar

precio = np.array([1.5, 2.0, 0.5])   # por producto
print(ventas * precio)               # (2,3) * (3,) -> por columna

meta = np.array([[25], [45]])        # por tienda
print(ventas - meta)                 # (2,3) - (2,1) -> por fila

[[ 20  40  60]
 [ 80 100 120]]
[[ 15.  40.  15.]
 [ 60. 100.  30.]]
[[-15  -5   5]
 [ -5   5  15]]


## 6. Agregaciones y el eje (`axis`)

Truco: `axis=0` significa que el resultado **ya no tiene filas** (ese eje se colapsó).

In [13]:
m = np.array([[10, 20, 30],
              [40, 50, 60]])

print(m.sum())          # todo el array
print(m.sum(axis=0))    # colapsa filas -> por columna
print(m.sum(axis=1))    # colapsa columnas -> por fila
print(m.mean(), m.std(), m.min(), m.max())
print(m.argmax())       # índice del máximo (aplanado)

210
[50 70 90]
[ 60 150]
35.0 17.07825127659933 10 60
5


## 7. Memoria: la transposición es gratis

In [14]:
m = np.arange(12).reshape(3, 4)
print(m.strides)    # bytes a saltar por eje
print(m.T.strides)  # transponer solo cambia los strides, no copia

(32, 8)
(8, 32)


## 8. Ejercicios

Completen donde dice `# ¿Qué va aquí?`. Regla: **sin ciclos `for`**.

### Ejercicio 1: z-score sin loops

Normalizar un array: restar la media y dividir entre la desviación estándar.

In [15]:
rng = np.random.default_rng(7)
alturas = rng.normal(170, 10, size=1000)

def z_score(x):
    # ¿Qué va aquí? (sin for)
    pass

z = z_score(alturas)
# Verificación (descomenten al terminar):
# print(round(z.mean(), 4), round(z.std(), 4))  # ~0 y ~1

### Ejercicio 2: distancias con broadcasting

Distancia euclidiana de cada punto a un centro, sin loops. Esta operación exacta es el corazón de k-NN y k-Means (semanas 6 y 11).

In [16]:
puntos = rng.normal(size=(500, 2))   # 500 puntos en 2D
centro = np.array([1.0, 1.0])

# ¿Qué va aquí?
# Pista: (puntos - centro) usa broadcasting (500,2) - (2,)
# Luego: elevar al cuadrado, sumar con axis=1, sacar raíz
distancias = ...

# ¿Cuántos puntos están a menos de 1 del centro?
cercanos = ...

# Verificación (descomenten al terminar):
# print(distancias.shape)  # (500,)
# print(cercanos)

### Ejercicio 3: análisis de notas

In [17]:
# Notas de 200 estudiantes en 3 parciales
notas = rng.integers(40, 101, size=(200, 3))

# a) promedio de cada estudiante (shape (200,))
promedio_estudiante = ...

# b) promedio de cada parcial (shape (3,))
promedio_parcial = ...

# c) ¿cuántos estudiantes aprobaron (promedio >= 61)?
aprobados = ...

# d) ¿qué estudiante (índice) tuvo el mejor promedio?
mejor = ...

## Lo esencial de hoy

- Un ndarray es un **bloque contiguo** de un solo `dtype`
- **Vectorizar**: las ufuncs corren en C; el `for` numérico es sospechoso
- **Máscaras booleanas** para filtrar; ojo con vistas vs copias
- **Broadcasting**: alinear desde la derecha, el 1 se estira
- `axis=k` colapsa el eje `k`; `%timeit` para medir

**Próxima clase (martes 28): Pandas y SQL.** Se asigna la HDT 1 (entrega: martes 4 de agosto).